# Log Probabilities

Based on [Probability for Computer Scientists — Log Probabilities](https://chrispiech.github.io/probabilityForComputerScientists/en/part1/log_probabilities/)

A **log probability** $\log P(E)$ is simply the $\log$ function applied to a probability.  
For example, if $P(E) = 0.00001$, then $\log P(E) = \log(0.00001) \approx -11.51$.

> Throughout this notebook we use the **natural logarithm** ($\ln$, base $e$).

### Why log probabilities?

| Reason | Explanation |
|--------|-------------|
| **Products → Addition** | Multiplying many small probabilities becomes summing their logs |
| **Numerical stability** | Computers cannot represent numbers smaller than $\approx 2.225 \times 10^{-308}$ (floating-point underflow) |
| **Computational speed** | Addition is faster than multiplication on modern hardware |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print('Setup complete!')

---
## 1 · The log‑probability number line

Since $0 \le P(E) \le 1$, taking the log maps this to the range:

$$-\infty \le \log P(E) \le 0$$

- $P(E) = 1  \;\Rightarrow\; \log P(E) = 0$ (certain event)
- $P(E) \to 0 \;\Rightarrow\; \log P(E) \to -\infty$ (impossible event)

Let's visualize the mapping from probability space $[0, 1]$ to log‑probability space $(-\infty, 0]$.

In [ ]:
# --- Visualize the log mapping ---
p = np.linspace(0.001, 1, 500)
log_p = np.log(p)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: the curve
ax = axes[0]
ax.plot(p, log_p, color='#1565C0', linewidth=2.5)
ax.axhline(0, color='grey', linewidth=0.5)
ax.set_xlabel('$P(E)$')
ax.set_ylabel('$\\log\, P(E)$')
ax.set_title('Mapping: probability → log probability')

# Annotate a few key points
for prob, label in [(1.0, '$P=1$'), (0.5, '$P=0.5$'), (0.01, '$P=0.01$')]:
    lp = np.log(prob)
    ax.plot(prob, lp, 'o', color='#C62828', markersize=7, zorder=5)
    ax.annotate(f'{label}\nlog ≈ {lp:.2f}', (prob, lp),
                textcoords='offset points', xytext=(12, 6),
                fontsize=9, color='#C62828')

# Right: number line comparison
ax = axes[1]
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Two equivalent representations')

# Probability line
ax.annotate('', xy=(1.3, 0.7), xytext=(-0.1, 0.7),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#1565C0'))
ax.text(0.6, 1.0, 'Probability space  $[0,\;1]$',
        ha='center', fontsize=10, color='#1565C0', weight='bold')
for v in [0, 0.25, 0.5, 0.75, 1.0]:
    x = -0.1 + v * 1.4
    ax.plot(x, 0.7, '|', color='#1565C0', markersize=10)
    ax.text(x, 0.45, f'{v}', ha='center', fontsize=8, color='#1565C0')

# Log-probability line
ax.annotate('', xy=(1.3, -0.5), xytext=(-0.1, -0.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#C62828'))
ax.text(0.6, -0.2, 'Log-probability space  $(-\\infty,\;0]$',
        ha='center', fontsize=10, color='#C62828', weight='bold')
for v, lbl in [(-10, '−∞'), (-5, ''), (-2, '−2'), (-1, '−1'), (0, '0')]:
    frac = (v + 10) / 10          # map [-10, 0] → [0, 1]
    x = -0.1 + frac * 1.4
    ax.plot(x, -0.5, '|', color='#C62828', markersize=10)
    ax.text(x, -0.75, lbl if lbl else '', ha='center', fontsize=8, color='#C62828')

plt.tight_layout()
plt.show()

---
## 2 · Products become Addition

One of the most useful properties of logarithms for probability:

$$\log\bigl(P(E) \cdot P(F)\bigr) = \log P(E) + \log P(F)$$

This generalises to any number of events:

$$\log \prod_i P(E_i) = \sum_i \log P(E_i)$$

### Why does this matter?

Suppose you flip a fair coin 100 times and each flip is independent.  
The probability of a specific sequence is:

$$P(\text{sequence}) = \prod_{i=1}^{100} P(\text{flip}_i) = 0.5^{100} \approx 7.9 \times 10^{-31}$$

In log space this is just:

$$\log P(\text{sequence}) = 100 \cdot \log(0.5) \approx -69.3$$

Much easier to store and compute!

In [ ]:
# --- Demonstrate: product vs sum in log space ---

n_flips = np.arange(1, 201)
p_fair = 0.5

# Direct probability (will underflow to 0 for large n)
prob_direct = p_fair ** n_flips

# Log probability (stays representable)
log_prob = n_flips * np.log(p_fair)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: direct probability
ax = axes[0]
ax.plot(n_flips, prob_direct, color='#C62828', linewidth=2)
ax.set_xlabel('Number of flips $n$')
ax.set_ylabel('$P(\\text{sequence}) = 0.5^n$')
ax.set_title('Direct probability — vanishes quickly')
ax.set_yscale('linear')
ax.axhline(0, color='grey', lw=0.5)

# Right: log probability
ax = axes[1]
ax.plot(n_flips, log_prob, color='#1565C0', linewidth=2)
ax.set_xlabel('Number of flips $n$')
ax.set_ylabel('$\\log P(\\text{sequence}) = n \\cdot \\log(0.5)$')
ax.set_title('Log probability — always representable')

plt.tight_layout()
plt.show()

print(f'Direct:  0.5^200  = {p_fair**200}')
print(f'Log:     200·log(0.5) = {200 * np.log(p_fair):.4f}  (easy to store!)')

---
## 3 · Visualizing products → addition

Let's take three independent events and compare multiplying vs adding in log space.

In [ ]:
# --- Side-by-side: product in probability space vs sum in log space ---
events = {'$P(A)$': 0.3, '$P(B)$': 0.4, '$P(C)$': 0.5}
names = list(events.keys())
probs = list(events.values())
log_probs = [np.log(p) for p in probs]

product = np.prod(probs)
log_sum = np.sum(log_probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#1565C0', '#2E7D32', '#EF6C00']

# Left: probability bars + product
ax = axes[0]
x = np.arange(len(names) + 1)
vals = probs + [product]
bar_colors = colors + ['#C62828']
labels = names + ['$P(A)·P(B)·P(C)$']
bars = ax.bar(x, vals, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Probability')
ax.set_title('Probability space — multiply')
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9, weight='bold')
ax.set_ylim(0, 0.65)

# Right: log probability bars + sum
ax = axes[1]
vals_log = log_probs + [log_sum]
bars = ax.bar(x, vals_log, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels([n.replace('P', '\\log P') for n in names] + ['$\\sum \\log P$'],
                    fontsize=9)
ax.set_ylabel('Log probability')
ax.set_title('Log space — add')
for i, v in enumerate(vals_log):
    ax.text(i, v - 0.12, f'{v:.3f}', ha='center', fontsize=9, weight='bold', va='top')

plt.tight_layout()
plt.show()

print(f'Product:     {probs[0]} × {probs[1]} × {probs[2]} = {product}')
print(f'Sum of logs: {log_probs[0]:.4f} + {log_probs[1]:.4f} + {log_probs[2]:.4f} = {log_sum:.4f}')
print(f'exp(sum)   = {np.exp(log_sum):.4f}  ✓ matches product')

---
## 4 · Representing Very Small Probabilities (Underflow)

Python's `float` uses IEEE 754 double-precision. The **smallest positive normal number** is:

$$\text{float\_min} \approx 2.225 \times 10^{-308}$$

Anything smaller **underflows to 0**. But its logarithm is perfectly representable:

$$\log(2.225 \times 10^{-308}) \approx -708.4$$

### Real-world example

Imagine computing the probability that an author wrote a specific 500-word document, where each word has probability $\approx 0.001$:

$$P(\text{document}) = \prod_{i=1}^{500} P(w_i) = 0.001^{500} = 10^{-1500}$$

This is **way** below $10^{-308}$ — the computer gives us $0$. But in log space:

$$\log P(\text{document}) = 500 \cdot \log(0.001) \approx -3454$$

Perfectly storable!

In [ ]:
# --- Demonstrate floating-point underflow ---
import sys

print(f'Smallest positive float:  {sys.float_info.min:.3e}')
print(f'Its log:                  {np.log(sys.float_info.min):.3f}')
print()

# Multiply many small probabilities
p_word = 0.001
n_words_list = [100, 200, 300, 500, 1000]

print(f'{"n words":>8} {"Direct prob":>20} {"Log prob":>15}')
print('-' * 50)
for n in n_words_list:
    direct = p_word ** n          # will underflow for large n
    logp = n * np.log(p_word)     # always representable
    direct_str = f'{direct:.3e}' if direct > 0 else '*** UNDERFLOW (0.0) ***'
    print(f'{n:>8} {direct_str:>20} {logp:>15.2f}')

In [ ]:
# --- Visualize the underflow cliff ---
n_words = np.arange(1, 400)
p_word = 0.01

direct_probs = np.array([p_word ** n for n in n_words])
log_probs_arr = n_words * np.log(p_word)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: direct — underflow cliff
ax = axes[0]
ax.semilogy(n_words, np.where(direct_probs > 0, direct_probs, np.nan),
            color='#C62828', linewidth=2)
underflow_n = n_words[direct_probs == 0]
if len(underflow_n) > 0:
    ax.axvline(underflow_n[0], color='grey', linestyle='--', linewidth=1)
    ax.text(underflow_n[0] + 5, 1e-200, f'Underflow at n={underflow_n[0]}',
            fontsize=9, color='grey')
ax.set_xlabel('Number of events $n$')
ax.set_ylabel('$P = 0.01^n$  (log scale y-axis)')
ax.set_title('Direct probability — hits underflow')

# Right: log — no underflow
ax = axes[1]
ax.plot(n_words, log_probs_arr, color='#1565C0', linewidth=2)
ax.set_xlabel('Number of events $n$')
ax.set_ylabel('$\\log P = n \cdot \\log(0.01)$')
ax.set_title('Log probability — no underflow risk')
ax.axhline(-708, color='grey', linestyle='--', linewidth=1)
ax.text(200, -708 + 40, 'log(float_min) ≈ −708', fontsize=9, color='grey')

plt.tight_layout()
plt.show()

---
## 5 · Practical example: comparing two authors

Suppose you have a document and want to decide which author (A or B) is more likely to have written it. Each author has a word-probability model. The document has $n$ words.

$$P_A(\text{doc}) = \prod_{i=1}^{n} P_A(w_i) \qquad P_B(\text{doc}) = \prod_{i=1}^{n} P_B(w_i)$$

We pick the author with the **higher** probability. In log space, we pick the author with the **less negative** log probability (closer to 0).

In [ ]:
# --- Simulated author comparison ---
np.random.seed(42)

n_words = 200
# Author A's word probabilities (slightly higher on average)
word_probs_A = np.random.uniform(0.0005, 0.01, n_words)
# Author B's word probabilities (slightly lower on average)
word_probs_B = np.random.uniform(0.0003, 0.008, n_words)

# Direct computation — will both underflow!
prob_A_direct = np.prod(word_probs_A)
prob_B_direct = np.prod(word_probs_B)

# Log-space computation — no problem
log_prob_A = np.sum(np.log(word_probs_A))
log_prob_B = np.sum(np.log(word_probs_B))

print('=== Direct probability ===')
print(f'  P_A(doc) = {prob_A_direct}   ← underflow!')
print(f'  P_B(doc) = {prob_B_direct}   ← underflow!')
print(f'  Winner:  cannot decide (both are 0)!')
print()
print('=== Log probability ===')
print(f'  log P_A(doc) = {log_prob_A:.2f}')
print(f'  log P_B(doc) = {log_prob_B:.2f}')
winner = 'A' if log_prob_A > log_prob_B else 'B'
print(f'  Winner:  Author {winner}  (log prob closer to 0)')

In [ ]:
# --- Visualize the running log probability as words are processed ---
cum_log_A = np.cumsum(np.log(word_probs_A))
cum_log_B = np.cumsum(np.log(word_probs_B))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, n_words + 1), cum_log_A, color='#1565C0', linewidth=2,
        label=f'Author A  (final = {log_prob_A:.1f})')
ax.plot(range(1, n_words + 1), cum_log_B, color='#C62828', linewidth=2,
        label=f'Author B  (final = {log_prob_B:.1f})')
ax.set_xlabel('Word index $i$')
ax.set_ylabel('$\\sum_{j=1}^{i} \\log P(w_j)$')
ax.set_title('Running log probability — Author comparison')
ax.legend(fontsize=10)
ax.axhline(-708, color='grey', linestyle=':', linewidth=1, alpha=0.5)
ax.text(n_words * 0.8, -708 + 15, 'log(float_min)', fontsize=8, color='grey')
plt.tight_layout()
plt.show()

---
## 6 · Summary

| Property | Probability space | Log-probability space |
|----------|------------------|-----------------------|
| Range | $[0,\; 1]$ | $(-\infty,\; 0]$ |
| Certain event | $P(E) = 1$ | $\log P(E) = 0$ |
| Combining independent events | $P(E) \cdot P(F)$ (multiply) | $\log P(E) + \log P(F)$ (add) |
| Many events | $\prod_i P(E_i)$ | $\sum_i \log P(E_i)$ |
| Underflow risk | Yes, when $P < 2.225 \times 10^{-308}$ | No |
| Convert back | — | $P(E) = e^{\log P(E)}$ |

### Key formula

$$\boxed{\log \prod_i P(E_i) \;=\; \sum_i \log P(E_i)}$$